# ingest_sheet

Ingest a photo of an ArUco board: decode the QR to learn which board it is,
resolve that board's saved ingest config, then detect and label the pieces.

- **Does:** decode -> `load_sheet_config_by_id` -> `SheetAruco.load_sheet`, then
  reports piece count, slot labels, and a labelled overlay on the rectified sheet.
- **Needs:** a photo of a board, and that board's config already on disk under
  `data/aruco_boards/{id}/` (written by `generate_board`).
- **Produces:** nothing on disk; results are shown inline for validation.
- **Recreate the data:** run `generate_board`, display a board on screen,
  photograph it with pieces placed, and point `PHOTO_FP` at that image.

White and green boards ingest through these same cells; the background mask
engages only for green/blue, and only because the resolved config carries it.
Why the mask matters: see
[docs/guides/green_background.md](../docs/guides/green_background.md). This
notebook only drives the pieces; it defines no logic of its own.


In [ ]:
import cv2
import matplotlib.pyplot as plt

from snap_fit.aruco.board_config_resolver import BoardConfigNotFoundError
from snap_fit.aruco.board_config_resolver import load_sheet_config_by_id
from snap_fit.aruco.sheet_metadata import SheetMetadataDecoder
from snap_fit.image.utils import load_image
from snap_fit.params.snap_fit_params import get_snap_fit_params
from snap_fit.puzzle.sheet_aruco import SheetAruco

In [ ]:
# --- parameters -----------------------------------------------------------
# Board photo to ingest, resolved from the project data folder (SnapFitPaths) so
# it works regardless of the notebook's working directory. The greendemo_v1 set
# is gitignored sample data; "straight" shots decode best. Point this at your
# own capture under data/.
sheets_fol = get_snap_fit_params().paths.data_fol / "greendemo_v1" / "sheets"
PHOTO_FP = sheets_fol / "greendemo_p1_1x_straight.jpg"
# PHOTO_FP = sheets_fol / "greendemo_p2_5x_side.jpg"  # harder angle

In [ ]:
# Decode the QR and resolve the stored config by board id (no config built here).
img = load_image(PHOTO_FP)
metadata = SheetMetadataDecoder().decode(img)
if metadata is None:
    msg = f"no QR metadata decoded from {PHOTO_FP.name}; cannot resolve a config"
    raise RuntimeError(msg)
print(f"decoded board_config_id={metadata.board_config_id!r}")

try:
    config = load_sheet_config_by_id(metadata.board_config_id)
except BoardConfigNotFoundError as exc:
    msg = (
        f"no saved config for {metadata.board_config_id!r}; "
        "run generate_board for this board first"
    )
    raise RuntimeError(msg) from exc
config.preprocess.background_mask

In [ ]:
# Detect and label the pieces using the resolved config.
sheet = SheetAruco(config).load_sheet(PHOTO_FP)
print(f"metadata: {sheet.metadata}")
print(f"pieces detected: {len(sheet.pieces)}")

In [ ]:
# Validate: piece -> slot label. A None label means the centroid fell off-grid.
for piece in sheet.pieces:
    print(f"  piece {piece.piece_id.piece_id:>3}  label={piece.label!r}")
labelled = sum(p.label is not None for p in sheet.pieces)
print(f"{labelled}/{len(sheet.pieces)} pieces got a slot label")

In [ ]:
# Labelled overlay on the rectified sheet (cropped-sheet space).
# centroid_in_sheet is the sheet-space accessor, so it lines up with img_orig.
fig, ax = plt.subplots(figsize=(8, 11))
ax.imshow(cv2.cvtColor(sheet.img_orig, cv2.COLOR_BGR2RGB))
for piece in sheet.pieces:
    cx, cy = piece.centroid_in_sheet
    ax.plot(cx, cy, "o", color="red", markersize=4)
    ax.annotate(
        piece.label or "?",
        (cx, cy),
        color="yellow",
        fontsize=12,
        ha="center",
        va="center",
    )
ax.set_title(f"{PHOTO_FP.name}: {len(sheet.pieces)} pieces")
ax.axis("off")
plt.tight_layout()
plt.show()